# UC1 — PCB (Printed Circuit Board): 1. Dataset Preparation

Before training or generation, the PCB data must be fetched and arranged
into the layout Cosmos AnomalyGen expects. This notebook does exactly that.

> **How commands run in this tutorial.** All pipeline steps run inside the
> `cosmos-predict2` conda environment. In a notebook cell we prefix shell
> commands with `conda run -n cosmos-predict2` (add `--live-stream` to stream
> logs live). If you prefer, open a JupyterLab **Terminal**, run
> `conda activate cosmos-predict2` once, and paste the same commands without
> the `conda run` prefix.
>
> If that environment does not exist yet, build it first with the top-level
> [tutorial/notebooks/0-setup-cuda128.ipynb](../../0-setup-cuda128.ipynb) — see
> the prerequisite note below.

## 1.0 Set the project root

In [ ]:
# Resolve the repository root (the folder containing pyproject.toml) and cd into it,
# so every relative path below (datasets/, checkpoints/, results/, scripts/) resolves.
import os
d = os.getcwd()
while d != "/" and not os.path.exists(os.path.join(d, "pyproject.toml")):
    d = os.path.dirname(d)
LOCAL_PROJECT_DIR = d
os.chdir(LOCAL_PROJECT_DIR)
# Pipeline scripts read the finetuned models & write outputs under the repo root.
os.environ.setdefault("IMAGINAIRE_OUTPUT_ROOT", "./results")
print("Project root:", LOCAL_PROJECT_DIR)

## 1.1 Where the data comes from

| UC | Subject | Anomaly types | HF provides | Local prep |
|---|---|---|---|---|
| UC1 | PCB | `IC+bridge`, `passive_component+excess_solder`, `passive_component+missing` | Full dataset on Hugging Face | `prepare_dataset_uc1.py` (auto-download) |

> **License notice.** You are responsible for confirming each dataset's license is
> fit for your intended use. See `datasets/README.md`.

## 1.2 Download & organize the dataset

Run the preparation script — it downloads and reorganizes the data into the exact
layout the trainer/generator expect.

In [ ]:
# Idempotent: skip if already prepared (the prep script is not safe to re-run
# into a populated directory). Delete datasets/UC1_pcb/ to force a fresh download.
!if [ -f datasets/UC1_pcb/defect_spec.jsonl ]; then \
   echo "datasets/UC1_pcb already prepared - skipping (delete it to re-download)."; \
 else \
   conda run -n cosmos-predict2 python -m scripts.utilities.prepare_dataset_uc1 datasets/UC1_pcb; \
 fi

## 1.3 Expected layout

After preparation, `datasets/UC1_pcb/` looks like this:

```
datasets/UC1_pcb/
  IC/
    anomaly_image/<TYPE>/   real defect images
    mask/<TYPE>/            paired binary masks (<stem>_mask.png)
    clean_image/            defect-free canvases for generation
    cad_mask/               CAD masks for cad2roi placement
  passive_component/
    anomaly_image/<TYPE>/   real defect images
    mask/<TYPE>/            paired binary masks (<stem>_mask.png)
    clean_image/            defect-free canvases for generation
    cad_mask/               CAD masks for cad2roi placement
  defect_spec.jsonl         one line per defect: type + spatial_dependency
  semantic_segmentation_labels.json   label map used by cad2roi
```

**Conventions the pipeline relies on:**
- `anomaly_types` are `[TEXTURE, TYPE]` pairs — the first element must match the texture folder name.
- Every anomaly image has a paired mask with the `_mask` suffix (`img_001.png` ↔ `img_001_mask.png`).
- `clean_image/` holds defect-free images used as canvases during generation.
- `cad_mask/` supplies the CAD-mask ROI for every `cad` defect (all UC1 defects, incl. `IC+bridge`); AMP needs it in notebook 3.

Inspect it:

In [ ]:
!find datasets/UC1_pcb -maxdepth 3 -type d | sort

## 1.4 The defect specification

`defect_spec.jsonl` tags each defect with a `spatial_dependency`
(`free` / `cad` / `text`) that controls how masks are placed during testcase
preparation (notebook 3).

In [ ]:
!cat datasets/UC1_pcb/defect_spec.jsonl

## 1.5 Preview a few samples

Overlay the real masks on the real anomaly images to sanity-check the pairing.

In [ ]:
import glob, os
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

dataset_dir = "datasets/UC1_pcb"
pairs = []
for anom in sorted(glob.glob(f"{dataset_dir}/*/anomaly_image/*/*")):
    stem = os.path.splitext(os.path.basename(anom))[0]
    mask_dir = os.path.dirname(anom).replace("/anomaly_image/", "/mask/")
    # mask lookup is extension-agnostic and tolerates the "_mask" suffix or none
    cand = (glob.glob(os.path.join(mask_dir, stem + "_mask.*"))
            or glob.glob(os.path.join(mask_dir, stem + ".*")))
    if cand:
        pairs.append((anom, cand[0]))
    if len(pairs) >= 3:
        break

if not pairs:
    print("No (image, mask) pairs found under", dataset_dir, "- run the preparation step above first.")
else:
    fig, axes = plt.subplots(len(pairs), 3, figsize=(10, 3.2 * len(pairs)))
    axes = np.atleast_2d(axes)
    for r, (a, m) in enumerate(pairs):
        img = Image.open(a).convert("RGB")
        # masks may be authored at the original resolution; align to the image for overlay
        msk = Image.open(m).convert("L").resize(img.size)
        ov = np.array(img).copy(); mk = np.array(msk) > 127
        ov[mk] = (0.5 * ov[mk] + np.array([255, 0, 0]) * 0.5).astype("uint8")
        for ax, im, t in zip(axes[r], [img, msk, Image.fromarray(ov)],
                             ["anomaly image", "mask", "overlay"]):
            ax.imshow(im); ax.set_title(f"{os.path.basename(a).rsplit(chr(46), 1)[0]} — {t}", fontsize=8); ax.axis("off")
    plt.tight_layout(); plt.show()

## Next Step

Proceed to [2-training.ipynb](./2-training.ipynb).